# Fine-tune & Benchmark: Alibaba-NLP/gte-multilingual-base

**Dataset**: Zalo AI Challenge 2021 — Legal Text Retrieval (3,298 Q–article pairs, corpus 61k điều luật).

**Pipeline**:
1. Load + split 80/20 (seed=42).
2. Baseline zero-shot eval (no fine-tune).
3. Fine-tune **in-batch negatives** (MultipleNegativesRankingLoss) + per-epoch eval.
4. Fine-tune **BM25 hard negatives** từ in-batch best checkpoint + per-epoch eval.
5. Pick overall winner → push lên HuggingFace Hub.

**Metrics**: F2@k (Zalo competition), Recall@k, Precision@k, MRR, nDCG@k với k ∈ {1, 5, 10}.

**Kaggle setup**:
- Accelerator: GPU T4 x2 (hoặc P100).
- Internet: ON.
- Add Data: Zalo Legal dataset (chứa `corpus.jsonl` + `qa.jsonl`).
- Add Secret: `HF_TOKEN` (dùng để push model).

**Est. time (5 epochs)**: ~1.4h trên T4.

## Cell 1 — Install dependencies

In [ ]:
# IMPORTANT: Set BEFORE any torch import. Kaggle T4 x2 → only use GPU 0
# to avoid DataParallel overhead with MultipleNegativesRankingLoss.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

!pip install -q sentence-transformers==3.3.1 datasets==2.20.0 rank_bm25==0.2.2 pyvi==0.1.1 huggingface_hub==0.26.2

## Cell 2 — Config

In [ ]:
# Config
MODEL_NAME = 'Alibaba-NLP/gte-multilingual-base'
NOTEBOOK_TAG = 'gte-multilingual-base'
N_EPOCHS = 5
BATCH_SIZE_TRAIN = 8  # single GPU + grad checkpoint
BATCH_SIZE_EVAL  = 8  # eval no backprop
MAX_SEQ_LENGTH = 384  # legal articles mostly < 384 tokens
TRUST_REMOTE_CODE = True
IS_E5_MODEL = False  # E5 cần prefix 'query: ' và 'passage: '
LR = 2e-5
SEED = 42
EARLY_STOPPING_PATIENCE = 5

# HuggingFace Hub — đổi 'username' thành tài khoản của bạn
HF_HUB_REPO = f"nhonhoccode/zalo-legal-{NOTEBOOK_TAG}-finetuned"
PUSH_TO_HUB = True  # set False nếu không muốn push

# Paths
import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # Kaggle: Accelerator = GPU T4 x2 + CUDA_VISIBLE_DEVICES set above
from pathlib import Path
WORKING = Path("/kaggle/working")
WORKING.mkdir(exist_ok=True)
INPUT_BASE = Path("/kaggle/input/datasets/nhondangcode/nlp-2026")


## Cell 3 — Imports + seed

In [ ]:
import json
import math
import random
import time
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import torch
from tqdm.auto import tqdm

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} | Count: {torch.cuda.device_count()}")

## Cell 4 — Load Zalo data

In [ ]:
# Auto-detect Zalo data files in Kaggle inputs
qa_files = list(INPUT_BASE.rglob("qa.jsonl"))
corpus_files = list(INPUT_BASE.rglob("corpus.jsonl"))
assert qa_files, "Upload Zalo dataset (qa.jsonl) to Kaggle Inputs first!"
assert corpus_files, "Upload Zalo dataset (corpus.jsonl) to Kaggle Inputs first!"
QA_FILE, CORPUS_FILE = qa_files[0], corpus_files[0]
print(f"QA:     {QA_FILE}")
print(f"Corpus: {CORPUS_FILE}")

# Load corpus
corpus = []
with CORPUS_FILE.open() as f:
    for line in f:
        if line.strip():
            corpus.append(json.loads(line))
print(f"Corpus articles: {len(corpus)}")

# Build (law_id, article_id) → text index + (law_id, article_id) → corpus_idx
corpus_key_to_idx = {}
corpus_texts = []
for i, art in enumerate(corpus):
    key = (str(art["law_id"]), str(art["article_id"]))
    corpus_key_to_idx[key] = i
    corpus_texts.append(art["text"])
print(f"Unique keys: {len(corpus_key_to_idx)}")

# Load Q&A
qa_all = []
with QA_FILE.open() as f:
    for line in f:
        if line.strip():
            qa_all.append(json.loads(line))
print(f"Total Q&A: {len(qa_all)}")

# Filter: chỉ giữ Q&A có ALL relevant_articles match được trong corpus.
qa_valid = []
n_dropped = 0
for q in qa_all:
    rel = q.get("relevant_articles") or []
    # corpus.jsonl dùng article_id = '{law_id}__{number}', qa.jsonl chỉ có '{number}'
    def _norm_key(r):
        lid, aid = str(r["law_id"]), str(r["article_id"])
        return (lid, aid if "__" in aid else f"{lid}__{aid}")
    keys = [_norm_key(r) for r in rel]
    if rel and all(k in corpus_key_to_idx for k in keys):
        qa_valid.append({**q, "relevant_keys": keys})
    else:
        n_dropped += 1
print(f"Valid Q&A: {len(qa_valid)} | Dropped (no matching article): {n_dropped}")

## Cell 5 — 80/20 split (deterministic)

In [ ]:
# 80/20 split — deterministic with SEED
from sklearn.model_selection import train_test_split

qa_train, qa_test = train_test_split(
    qa_valid, test_size=0.2, random_state=SEED, shuffle=True,
)
print(f"Train: {len(qa_train)} | Test: {len(qa_test)}")

# Save split để các notebook khác dùng cùng test set
split_meta = {
    "seed": SEED,
    "train_ids": [q["question_id"] for q in qa_train],
    "test_ids":  [q["question_id"] for q in qa_test],
}
split_path = WORKING / "zalo_split.json"
with split_path.open("w") as f:
    json.dump(split_meta, f)
print(f"Saved split: {split_path}")

## Cell 6 — Inline metrics (F2, Recall, MRR, nDCG)

In [ ]:
# Metrics inline (port từ backend/src/eval/retrieval_metrics.py + thêm F2)
# Input: hits = list[(law_id, article_id)] đã sort theo score desc.
#        relevant = set[(law_id, article_id)]

def recall_at_k(hits, relevant, k):
    if not relevant: return 0.0
    return float(any(h in relevant for h in hits[:k]))

def precision_at_k(hits, relevant, k):
    if not relevant or k <= 0: return 0.0
    top_k = hits[:k]
    if not top_k: return 0.0
    return sum(1 for h in top_k if h in relevant) / k

def reciprocal_rank(hits, relevant):
    if not relevant: return 0.0
    for rank, h in enumerate(hits, 1):
        if h in relevant: return 1.0 / rank
    return 0.0

def ndcg_at_k(hits, relevant, k):
    if not relevant or k <= 0: return 0.0
    top_k = hits[:k]
    dcg = sum(1.0/math.log2(i+1) for i, h in enumerate(top_k, 1) if h in relevant)
    n_rel = min(len(relevant), k)
    idcg = sum(1.0/math.log2(i+1) for i in range(1, n_rel+1))
    return dcg/idcg if idcg > 0 else 0.0

def f_beta_at_k(hits, relevant, k, beta=2.0):
    """F-beta@k. β=2 → Zalo AI Challenge 2021 metric."""
    if not relevant or k <= 0: return 0.0
    matched = sum(1 for h in hits[:k] if h in relevant)
    p = matched / k
    r = matched / len(relevant)
    if p == 0 and r == 0: return 0.0
    b2 = beta * beta
    return (1 + b2) * p * r / (b2 * p + r)

def compute_metrics(predictions, gold_lists, k_values=(1, 5, 10)):
    """predictions: list[list[(law_id, article_id)]]; gold_lists: list[set[...]]."""
    n = len(predictions)
    assert n == len(gold_lists)
    out = {f"recall@{k}": 0.0 for k in k_values}
    out.update({f"precision@{k}": 0.0 for k in k_values})
    out.update({f"ndcg@{k}": 0.0 for k in k_values})
    out.update({f"f2@{k}": 0.0 for k in k_values})
    out["mrr"] = 0.0
    for hits, gold in zip(predictions, gold_lists):
        out["mrr"] += reciprocal_rank(hits, gold)
        for k in k_values:
            out[f"recall@{k}"] += recall_at_k(hits, gold, k)
            out[f"precision@{k}"] += precision_at_k(hits, gold, k)
            out[f"ndcg@{k}"] += ndcg_at_k(hits, gold, k)
            out[f"f2@{k}"] += f_beta_at_k(hits, gold, k, beta=2.0)
    for k_ in list(out.keys()):
        out[k_] = round(out[k_] / n, 4)
    return out

## Cell 7 — Encode + retrieve helpers

In [ ]:
def encode_corpus(model, batch_size=64, query_prefix="", passage_prefix=""):
    """Encode toàn bộ corpus với optional prefix (e.g., E5 'passage: ')."""
    texts = [passage_prefix + t for t in corpus_texts] if passage_prefix else corpus_texts
    return model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )

def encode_queries(model, queries, batch_size=64, query_prefix=""):
    texts = [query_prefix + q for q in queries] if query_prefix else queries
    return model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=False,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )

def retrieve_top_k(query_vecs, corpus_vecs, k=10):
    """Cosine similarity (vecs đã normalize) → top-k indices per query."""
    sims = query_vecs @ corpus_vecs.T   # (n_q, n_corpus)
    top_idx = np.argpartition(-sims, kth=k, axis=1)[:, :k]
    # sort những top-k này theo score
    rows = np.arange(len(query_vecs))[:, None]
    sorted_order = np.argsort(-sims[rows, top_idx], axis=1)
    top_sorted = top_idx[rows, sorted_order]
    return top_sorted   # shape (n_q, k)

def eval_model(model, queries, gold_lists, *, query_prefix="", passage_prefix="", k_values=(1, 5, 10)):
    """End-to-end eval: encode corpus + queries → top-K → metrics."""
    t0 = time.time()
    corpus_vecs = encode_corpus(model, batch_size=BATCH_SIZE_EVAL, passage_prefix=passage_prefix)
    t_corpus = time.time() - t0

    t1 = time.time()
    q_vecs = encode_queries(model, queries, batch_size=BATCH_SIZE_EVAL, query_prefix=query_prefix)
    t_query = time.time() - t1

    top_idx = retrieve_top_k(q_vecs, corpus_vecs, k=max(k_values))
    predictions = [
        [(str(corpus[idx]["law_id"]), str(corpus[idx]["article_id"])) for idx in row]
        for row in top_idx
    ]
    metrics = compute_metrics(predictions, gold_lists, k_values=k_values)
    metrics["_t_corpus_encode_s"] = round(t_corpus, 1)
    metrics["_t_query_encode_s"] = round(t_query, 1)
    return metrics, predictions

## Cell 8 — Baseline zero-shot eval

In [ ]:
# ===== STEP 1: Baseline zero-shot eval (pretrained, no fine-tune) =====
from sentence_transformers import SentenceTransformer

print(f"Loading {MODEL_NAME}...")
load_kwargs = {"device": device}
if TRUST_REMOTE_CODE:
    load_kwargs["trust_remote_code"] = True
model = SentenceTransformer(MODEL_NAME, **load_kwargs)
model.max_seq_length = MAX_SEQ_LENGTH
# Enable gradient checkpointing: trade 20% speed for ~50% activation memory
try:
    model[0].auto_model.gradient_checkpointing_enable()
    print("Gradient checkpointing: ON")
except Exception as e:
    print(f"Gradient checkpointing not available: {e}")
print(f"Loaded. Dim={model.get_sentence_embedding_dimension()}")

# E5 prefix convention
q_prefix = "query: " if IS_E5_MODEL else ""
p_prefix = "passage: " if IS_E5_MODEL else ""

test_queries = [q["question"] for q in qa_test]
test_gold = [set(q["relevant_keys"]) for q in qa_test]

print("\n=== Baseline zero-shot eval ===")
baseline_metrics, _ = eval_model(
    model, test_queries, test_gold,
    query_prefix=q_prefix, passage_prefix=p_prefix,
)
print(json.dumps(baseline_metrics, indent=2))

(WORKING / "metrics_baseline.json").write_text(json.dumps(baseline_metrics, indent=2))

## Cell 9 — Fine-tune (in-batch negatives) + per-epoch eval

In [ ]:
# ===== STEP 2: Fine-tune with in-batch negatives + per-epoch eval =====
from sentence_transformers import losses
from sentence_transformers.readers import InputExample
from torch.utils.data import DataLoader

# Build train examples: (anchor=query, positive=article_text)
# E5 needs prefixes on both anchor and positive.
train_examples = []
for q in qa_train:
    query_text = q_prefix + q["question"]
    # 1 positive per question
    pos_key = q["relevant_keys"][0]
    pos_idx = corpus_key_to_idx[pos_key]
    pos_text = p_prefix + corpus_texts[pos_idx]
    train_examples.append(InputExample(texts=[query_text, pos_text]))

print(f"In-batch train pairs: {len(train_examples)}")

train_loader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE_TRAIN)
train_loss = losses.MultipleNegativesRankingLoss(model)

steps_per_epoch = len(train_loader)
print(f"Steps/epoch: {steps_per_epoch}")

# Manual training loop để có per-epoch eval + early stopping
inbatch_per_epoch = []
best_epoch = 0
best_score = -1.0
patience_left = EARLY_STOPPING_PATIENCE
best_checkpoint_dir = WORKING / "checkpoint_inbatch_best"
best_checkpoint_dir.mkdir(exist_ok=True)

for epoch in range(1, N_EPOCHS + 1):
    print(f"\n--- Epoch {epoch}/{N_EPOCHS} ---")
    t_train = time.time()
    model.fit(
        train_objectives=[(train_loader, train_loss)],
        epochs=1,
        warmup_steps=int(0.1 * steps_per_epoch) if epoch == 1 else 0,
        optimizer_params={"lr": LR},
        show_progress_bar=True,
        use_amp=True,
    )
    dt_train = time.time() - t_train

    metrics, _ = eval_model(
        model, test_queries, test_gold,
        query_prefix=q_prefix, passage_prefix=p_prefix,
    )
    metrics["epoch"] = epoch
    metrics["_t_train_s"] = round(dt_train, 1)
    inbatch_per_epoch.append(metrics)
    score = metrics["f2@10"]
    print(f"Epoch {epoch}: F2@10={score:.4f} | Recall@10={metrics['recall@10']:.4f} | MRR={metrics['mrr']:.4f} | train_s={dt_train:.0f}")

    if score > best_score:
        best_score = score
        best_epoch = epoch
        patience_left = EARLY_STOPPING_PATIENCE
        model.save(str(best_checkpoint_dir))
        print(f"  ✓ New best F2@10={score:.4f} — checkpoint saved.")
    else:
        patience_left -= 1
        print(f"  No improvement. Patience left: {patience_left}")
        if patience_left == 0:
            print(f"\n⚠️ Early stopping at epoch {epoch}.")
            break

inbatch_summary = {
    "per_epoch": inbatch_per_epoch,
    "best_epoch": best_epoch,
    "best_metrics": inbatch_per_epoch[best_epoch - 1] if best_epoch else {},
}
(WORKING / "metrics_inbatch.json").write_text(json.dumps(inbatch_summary, indent=2))
print(f"\n=== In-batch best: epoch {best_epoch}, F2@10={best_score:.4f} ===")

## Cell 10 — BM25 hard negatives mining

In [ ]:
# ===== STEP 3: BM25 hard negatives mining =====
from rank_bm25 import BM25Okapi
from pyvi import ViTokenizer

def tokenize_vi(text):
    return ViTokenizer.tokenize(text.lower()).split()

print("Building BM25 index over corpus...")
t0 = time.time()
tokenized_corpus = [tokenize_vi(t) for t in tqdm(corpus_texts, desc="Tokenize")]
bm25 = BM25Okapi(tokenized_corpus)
print(f"BM25 index built in {time.time()-t0:.1f}s")

# Mine hard negatives: BM25 top-30 → loại positive → 3-5 hard negatives
N_HARD = 5
BM25_TOP = 30
hard_neg_examples = []
n_skipped = 0
for q in tqdm(qa_train, desc="Mine hard negs"):
    pos_key = q["relevant_keys"][0]
    pos_idx = corpus_key_to_idx[pos_key]
    pos_text = corpus_texts[pos_idx]

    tokens = tokenize_vi(q["question"])
    scores = bm25.get_scores(tokens)
    top_idx = np.argpartition(-scores, kth=BM25_TOP)[:BM25_TOP]
    top_idx = top_idx[np.argsort(-scores[top_idx])]

    # Lọc bỏ positive
    hard_negs = []
    for idx in top_idx:
        if int(idx) == pos_idx:
            continue
        hard_negs.append(corpus_texts[int(idx)])
        if len(hard_negs) >= N_HARD:
            break

    if not hard_negs:
        n_skipped += 1
        continue

    # MultipleNegativesRankingLoss với 3-tuple+: [anchor, pos, hard_neg_1, ...]
    texts = [q_prefix + q["question"], p_prefix + pos_text]
    texts.extend(p_prefix + n for n in hard_negs)
    hard_neg_examples.append(InputExample(texts=texts))

print(f"Hard-neg pairs: {len(hard_neg_examples)} | Skipped: {n_skipped}")

## Cell 11 — Fine-tune (hard negatives, warm start) + per-epoch eval

In [ ]:
# ===== STEP 4: Fine-tune with hard negatives (warm start từ inbatch best) =====
print("Loading in-batch best checkpoint as warm start...")
model = SentenceTransformer(str(best_checkpoint_dir), device=device, trust_remote_code=TRUST_REMOTE_CODE)
model.max_seq_length = MAX_SEQ_LENGTH

# Hard-neg loader (batch nhỏ hơn vì mỗi sample có nhiều text)
HN_BATCH_SIZE = max(4, BATCH_SIZE_TRAIN // 2)
hn_loader = DataLoader(hard_neg_examples, shuffle=True, batch_size=HN_BATCH_SIZE)
hn_loss = losses.MultipleNegativesRankingLoss(model)

hn_steps_per_epoch = len(hn_loader)
print(f"Hard-neg steps/epoch: {hn_steps_per_epoch}, batch={HN_BATCH_SIZE}")

hardneg_per_epoch = []
hn_best_epoch = 0
hn_best_score = -1.0
hn_patience_left = EARLY_STOPPING_PATIENCE
hn_best_dir = WORKING / "checkpoint_hardneg_best"
hn_best_dir.mkdir(exist_ok=True)

for epoch in range(1, N_EPOCHS + 1):
    print(f"\n--- Hard-neg Epoch {epoch}/{N_EPOCHS} ---")
    t_train = time.time()
    model.fit(
        train_objectives=[(hn_loader, hn_loss)],
        epochs=1,
        warmup_steps=int(0.1 * hn_steps_per_epoch) if epoch == 1 else 0,
        optimizer_params={"lr": LR / 2},  # smaller LR for fine-tune step
        show_progress_bar=True,
        use_amp=True,
    )
    dt_train = time.time() - t_train

    metrics, _ = eval_model(
        model, test_queries, test_gold,
        query_prefix=q_prefix, passage_prefix=p_prefix,
    )
    metrics["epoch"] = epoch
    metrics["_t_train_s"] = round(dt_train, 1)
    hardneg_per_epoch.append(metrics)
    score = metrics["f2@10"]
    print(f"HN Epoch {epoch}: F2@10={score:.4f} | Recall@10={metrics['recall@10']:.4f} | train_s={dt_train:.0f}")

    if score > hn_best_score:
        hn_best_score = score
        hn_best_epoch = epoch
        hn_patience_left = EARLY_STOPPING_PATIENCE
        model.save(str(hn_best_dir))
        print(f"  ✓ New best F2@10={score:.4f} — checkpoint saved.")
    else:
        hn_patience_left -= 1
        print(f"  No improvement. Patience left: {hn_patience_left}")
        if hn_patience_left == 0:
            print(f"\n⚠️ Hard-neg early stopping at epoch {epoch}.")
            break

hardneg_summary = {
    "per_epoch": hardneg_per_epoch,
    "best_epoch": hn_best_epoch,
    "best_metrics": hardneg_per_epoch[hn_best_epoch - 1] if hn_best_epoch else {},
}
(WORKING / "metrics_hardneg.json").write_text(json.dumps(hardneg_summary, indent=2))
print(f"\n=== Hard-neg best: epoch {hn_best_epoch}, F2@10={hn_best_score:.4f} ===")

## Cell 12 — Final summary + winner

In [ ]:
# ===== STEP 5: Final summary — pick winner =====
candidates = [
    ("baseline", baseline_metrics, None),
    ("inbatch",  inbatch_summary["best_metrics"], inbatch_summary["best_epoch"]),
    ("hardneg",  hardneg_summary["best_metrics"], hardneg_summary["best_epoch"]),
]
winner_strategy, winner_metrics, winner_epoch = max(
    candidates, key=lambda x: x[1].get("f2@10", 0.0),
)

winner_dir_map = {
    "baseline": None,  # baseline = pretrained, không có checkpoint local
    "inbatch":  best_checkpoint_dir,
    "hardneg":  hn_best_dir,
}
winner_dir = winner_dir_map[winner_strategy]

final_summary = {
    "model_name": MODEL_NAME,
    "tag": NOTEBOOK_TAG,
    "baseline": baseline_metrics,
    "inbatch": inbatch_summary,
    "hardneg": hardneg_summary,
    "winner_strategy": winner_strategy,
    "winner_epoch": winner_epoch,
    "winner_metrics": winner_metrics,
    "config": {
        "n_epochs": N_EPOCHS,
        "batch_size_train": BATCH_SIZE_TRAIN,
        "lr": LR,
        "max_seq_length": MAX_SEQ_LENGTH,
        "n_train": len(qa_train),
        "n_test": len(qa_test),
        "seed": SEED,
    },
}
final_path = WORKING / f"metrics_final_{NOTEBOOK_TAG}.json"
final_path.write_text(json.dumps(final_summary, indent=2))
print(f"\n{'='*60}")
print(f"WINNER: {winner_strategy} (epoch={winner_epoch})")
print(f"F2@10:    {winner_metrics.get('f2@10', 0):.4f}")
print(f"Recall@10: {winner_metrics.get('recall@10', 0):.4f}")
print(f"MRR:      {winner_metrics.get('mrr', 0):.4f}")
print(f"Saved:    {final_path}")
print(f"{'='*60}")

## Cell 13 — Plot convergence curve

In [ ]:
# Plot per-epoch F2@10 convergence
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
if inbatch_summary["per_epoch"]:
    eps = [m["epoch"] for m in inbatch_summary["per_epoch"]]
    f2 = [m["f2@10"] for m in inbatch_summary["per_epoch"]]
    ax.plot(eps, f2, marker="o", label="In-batch negatives")
if hardneg_summary["per_epoch"]:
    eps = [m["epoch"] for m in hardneg_summary["per_epoch"]]
    f2 = [m["f2@10"] for m in hardneg_summary["per_epoch"]]
    ax.plot(eps, f2, marker="s", label="Hard negatives (warm start)")
ax.axhline(baseline_metrics["f2@10"], color="gray", linestyle="--", label=f"Baseline ({baseline_metrics['f2@10']:.3f})")
ax.set_xlabel("Epoch")
ax.set_ylabel("F2@10")
ax.set_title(f"Fine-tune convergence: {MODEL_NAME}")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(WORKING / f"convergence_{NOTEBOOK_TAG}.png", dpi=120)
plt.show()

In [ ]:
from kaggle_secrets import UserSecretsClient
secret = UserSecretsClient().get_secret("HF_TOKEN")
secret

## Cell 14 — Push winner to HF Hub

In [ ]:
# ===== STEP 6: Push winner checkpoint to HuggingFace Hub =====
if PUSH_TO_HUB and winner_dir is not None:
    from huggingface_hub import login
    try:
        from kaggle_secrets import UserSecretsClient
        secret = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        print(f"Kaggle secret HF_TOKEN không có ({e}). Skip push.")
        secret = None

    if secret:
        login(token=secret)
        print(f"Loading winner checkpoint from {winner_dir}...")
        win_model = SentenceTransformer(str(winner_dir), device=device, trust_remote_code=TRUST_REMOTE_CODE)
        win_model.max_seq_length = MAX_SEQ_LENGTH
        print(f"Pushing to {HF_HUB_REPO}...")
        win_model.push_to_hub(HF_HUB_REPO, private=True, exist_ok=True)
        print(f"✓ Pushed: https://huggingface.co/{HF_HUB_REPO}")
else:
    print(f"Skipped push. winner_strategy={winner_strategy}, PUSH_TO_HUB={PUSH_TO_HUB}")

# Backup zip
import shutil
if winner_dir is not None:
    zip_path = WORKING / f"checkpoint_{NOTEBOOK_TAG}_winner"
    shutil.make_archive(str(zip_path), "zip", winner_dir)
    print(f"Backup zip: {zip_path}.zip")

## Cell 15 — Sanity check

In [ ]:
# ===== Sanity check =====
sample_q = "Người lao động bị sa thải trái pháp luật được bồi thường gì?"
if winner_dir is not None:
    san_model = SentenceTransformer(str(winner_dir), device=device, trust_remote_code=TRUST_REMOTE_CODE)
else:
    san_model = model  # pretrained fallback
san_model.max_seq_length = MAX_SEQ_LENGTH

q_vec = encode_queries(san_model, [sample_q], query_prefix=q_prefix)
print("Encoding corpus for sanity (slower)...")
c_vec = encode_corpus(san_model, batch_size=BATCH_SIZE_EVAL, passage_prefix=p_prefix)
top_idx = retrieve_top_k(q_vec, c_vec, k=3)[0]

print(f"\nSample query: {sample_q}")
for rank, idx in enumerate(top_idx, 1):
    art = corpus[int(idx)]
    print(f"\n--- TOP {rank} ---")
    print(f"law_id: {art['law_id']} | article_id: {art['article_id']}")
    print(f"Text: {art['text'][:200]}...")